# OpenOBB: train nhanh trên Colab (CLI)

Notebook gọn dùng hằng ngày: mount Drive → cài OpenOBB → giải nén dataset → `openobb train` → tải `best.pt`.

- Chỉ cần sửa ô **3** (chọn dataset) và ô **4** (model, epoch, imgsz...).
- Mỗi lần train tạo thư mục mới `runs/obb/train`, `train2`... (giống YOLO). Weights ở `weights/best.pt`, `weights/last.pt`.
- Mất phiên giữa chừng: chạy lại ô 1–3 rồi `!openobb train resume=True`.
- GPU: *Runtime → Change runtime type* → **G4** (High-RAM).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Cài OpenOBB (bản mới nhất trên GitHub)

In [ ]:
!pip install -q --force-reinstall --no-deps git+https://github.com/Nguyenchitrai62/OpenOBB.git
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import openobb; print("OpenOBB", openobb.__version__)

## 3. Giải nén dataset và tạo `data.yaml`

Đổi `DATASET` để chọn. Mỗi dataset giải nén vào thư mục riêng, nên đổi qua lại trong cùng phiên không bị lẫn `data.yaml`.

In [ ]:
import glob, os, yaml

ZIPS = {
    "wall_color": "/content/drive/MyDrive/Dataset/tu_ph/Wall_Color/data/dataset_wall_color_v3.zip",
    "tivi":       "/content/drive/MyDrive/Dataset/tu_ph/Tivi/data_tivi_v2.zip",
    "door_foot":  "/content/drive/MyDrive/Dataset/tu_ph/Door_Foot/data_train.zip",
    "floor_plan": "/content/drive/MyDrive/Dataset/Floor_plan/dataset_obb_train_v3.zip",
}
DATASET = "floor_plan"

dst = f"/content/dataset_local/{DATASET}"
if not os.path.exists(dst + "/.unzipped"):            # chỉ giải nén một lần mỗi phiên
    !unzip -q -o "{ZIPS[DATASET]}" -d "{dst}"
    open(dst + "/.unzipped", "w").close()

# data.yaml nông nhất trong zip, ghi lại với 'path' tuyệt đối
y = min(glob.glob(f"{dst}/**/data.yaml", recursive=True), key=len)
cfg = {**yaml.safe_load(open(y)), "path": os.path.dirname(y)}
DATA = "/content/data.yaml"
yaml.safe_dump(cfg, open(DATA, "w"), allow_unicode=True, sort_keys=False)
print(open(DATA).read())

## 4. Train

- `MODEL`: tên kiến trúc (`openobb23x`, `openobb24x`...) để train mới, hoặc đường dẫn `.pt` để fine-tune tiếp
  (ví dụ `best.pt` của lần train trước, hoặc checkpoint trên Drive).
- Khung train giống YOLO: `lr` truyền vào là dùng đúng giá trị đó; bỏ `lr` thì tự đặt `0.002·5/(4+nc)`.
- `compile=True` đã là mặc định cho v18–v24, không cần truyền.
- **Không truyền `rotate_p`** (mặc định 0, giống `degrees=0` của YOLO): xoay ảnh làm box vật dài lệch hướng (F110).

In [ ]:
MODEL  = "/content/drive/MyDrive/Train/tu_ph/Wall_Color/model/openobb23x_new.pt"
# MODEL = "/content/runs/obb/train/weights/best.pt"     # fine-tune tiếp từ run vừa xong
# MODEL = "openobb23x"                                   # train mới
EPOCHS = 30
IMGSZ  = 960
BATCH  = 32
LR     = 0.001

!openobb train data={DATA} model={MODEL} epochs={EPOCHS} imgsz={IMGSZ} batch={BATCH} lr={LR}

## 5. Lấy `best.pt` của run mới nhất

Tải về máy, và chép lên Drive nếu đặt `DRIVE_OUT` (file vài trăm MB, chép Drive ổn định hơn tải qua trình duyệt).

In [ ]:
import shutil
from google.colab import files

DRIVE_OUT = None   # ví dụ "/content/drive/MyDrive/Train/tu_ph/Wall_Color/model/openobb23x_new2.pt"

bests = glob.glob("/content/runs/obb/train*/weights/best.pt")
if not bests:
    print("Không tìm thấy best.pt nào trong /content/runs/obb/")
else:
    best = max(bests, key=os.path.getmtime)
    print("best.pt mới nhất:", best)
    if DRIVE_OUT:
        os.makedirs(os.path.dirname(DRIVE_OUT), exist_ok=True)
        shutil.copy2(best, DRIVE_OUT)
        print("Đã chép lên Drive:", DRIVE_OUT)
    files.download(best)